In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

def fetch_all_studies(condition, page_size=1000, max_page=50):
    """Download every study matching a condition, and download one page at a time"""
    params = {"query.cond": condition, "pageSize": page_size}
    studies = []
    for page in range(max_page):
        response = requests.get(BASE_URL, timeout=60, params=params)
        response.raise_for_status()
        data = response.json()
        studies.extend(data.get("studies",[]))
        print(f"Page {page+1}: {len(studies)} studies so far")
        token = data.get("nextPageToken")
        if token is None:
            break
        params["pageToken"] = token
        time.sleep(1)
    return studies

In [4]:
studies = fetch_all_studies("non-small cell lung cancer") #Run once
print(len(studies))

Page 1: 1000 studies so far
Page 2: 2000 studies so far
Page 3: 3000 studies so far
Page 4: 4000 studies so far
Page 5: 5000 studies so far
Page 6: 6000 studies so far
Page 7: 7000 studies so far
Page 8: 8000 studies so far
Page 9: 8636 studies so far
8636


In [6]:
Path("../data/raw").mkdir(parents=True, exist_ok=True)

with open("../data/raw/nsclc_studies.json", "w") as f:
    json.dump(studies, f)

In [7]:
with open("../data/raw/nsclc_studies.json") as f:
    studies = json.load(f)
print(len(studies))

8636


In [ ]:
def flatten_study(study):
    ps = study.get("protocolSection", {})
    ident = ps.get("identificationModule", {})
    status = ps.get("statusModule", {})
    design = ps.get("designModule", {})
    sponsor = ps.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})
    return {
        "nct_id": ident.get("nctId"),
        "title": ident.get("briefTitle"),
        "status": status.get("overallStatus"),
        "start_date": status.get("startDateStruct", {}).get("date"),
        "why_stopped": status.get("whyStopped"),
        "study_type": design.get("studyType"),
        "phases": ",".join(design.get("phases", [])),
        "enrollment": design.get("enrollmentInfo", {}).get("count"),
        "sponsor": sponsor.get("name"),
        "sponsor_class": sponsor.get("class"),
        "has_results": study.get("hasResults"),}
    

In [15]:
df = pd.DataFrame([flatten_study(s) for s in studies])
df.to_csv("../data/processed/nsclc_trials_raw.csv", index=False)
print(df.shape)
df.head()

(8636, 11)


,nct_id,title,status,start_date,why_stopped,study_type,phases,enrollment,sponsor,sponsor_class,has_results
0,NCT00266877,Study Evaluating the Safety Of HKI-272 (Nerati...,COMPLETED,2005-12,NaN,INTERVENTIONAL,PHASE2,172.0,"Puma Biotechnology, Inc.",INDUSTRY,True
1,NCT04207775,Real World Study in Locally Advanced or Metast...,COMPLETED,2020-03-30,NaN,OBSERVATIONAL,,300.0,AstraZeneca,INDUSTRY,False
2,NCT00126581,Erlotinib Hydrochloride With or Without Carbop...,COMPLETED,2005-08-15,NaN,INTERVENTIONAL,PHASE2,188.0,National Cancer Institute (NCI),NIH,True
3,NCT05764928,Evaluate the Safety and Efficacy of BLEX 404 O...,NOT_YET_RECRUITING,2026-12-01,NaN,INTERVENTIONAL,"PHASE1,PHASE2",32.0,Rgene Corporation,INDUSTRY,False
4,NCT04450732,Safety of GQ1001 in Adult Patients With HER2-P...,UNKNOWN,2020-07-07,NaN,INTERVENTIONAL,PHASE1,96.0,"GeneQuantum Healthcare (Suzhou) Co., Ltd.",INDUSTRY,False
